# AI-Assisted Dashboard for Safety Signal Monitoring  
## Python: Data Loading & Validation

**Purpose of this notebook**  
This notebook focuses on loading, validating, and preparing a pharmacovigilance-style dataset
for downstream SQL analysis and Power BI visualization.

#Note:
- Detailed trend analysis is performed in SQL
- Visualization and forecasting are performed in Power BI
- Dataset is not included in this repository due to size constraints

In [3]:
#import libraries
import numpy as np 
import pandas as pd
# load dataset
df = pd.read_csv("drugs_side_effects_drugs_com.csv")

# preview data
df.head()

,drug_name,medical_condition,side_effects,generic_name,drug_classes,brand_names,activity,rx_otc,pregnancy_category,csa,...,Unnamed: 125,Unnamed: 126,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134
0,doxycycline,Acne,"(hives, difficult breathing, swelling in your ...",doxycycline,"Miscellaneous antimalarials, Tetracyclines","Acticlate, Adoxa CK, Adoxa Pak, Adoxa TT, Alod...",87%,Rx,D,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,spironolactone,Acne,hives ; difficulty breathing; swelling of your...,spironolactone,"Aldosterone receptor antagonists, Potassium-sp...","Aldactone, CaroSpir",82%,Rx,C,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,minocycline,Acne,"skin rash, fever, swollen glands, flu-like sym...",minocycline,Tetracyclines,"Dynacin, Minocin, Minolira, Solodyn, Ximino, V...",48%,Rx,D,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Accutane,Acne,problems with your vision or hearing; muscle o...,isotretinoin (oral),"Miscellaneous antineoplastics, Miscellaneous u...",NaN,41%,Rx,X,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,clindamycin,Acne,hives ; difficult breathing; swelling of your ...,clindamycin topical,"Topical acne agents, Vaginal anti-infectives","Cleocin T, Clindacin ETZ, Clindacin P, Clindag...",39%,Rx,B,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The dataset was reviewed only for structural integrity and completeness.
No exploratory visualization was performed at this stage.

In [5]:
df.shape

(2966, 135)

In [3]:
df.columns

Index(['drug_name', 'medical_condition', 'side_effects', 'generic_name',
       'drug_classes', 'brand_names', 'activity', 'rx_otc',
       'pregnancy_category', 'csa',
       ...
       'Unnamed: 125', 'Unnamed: 126', 'Unnamed: 127', 'Unnamed: 128',
       'Unnamed: 129', 'Unnamed: 130', 'Unnamed: 131', 'Unnamed: 132',
       'Unnamed: 133', 'Unnamed: 134'],
      dtype='object', length=135)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2966 entries, 0 to 2965
Columns: 135 entries, drug_name to Unnamed: 134
dtypes: float64(3), object(132)
memory usage: 3.1+ MB


In [5]:
# remove unnamed / empty columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# check result
df.columns

Index(['drug_name', 'medical_condition', 'side_effects', 'generic_name',
       'drug_classes', 'brand_names', 'activity', 'rx_otc',
       'pregnancy_category', 'csa', 'alcohol', 'related_drugs',
       'medical_condition_description', 'rating', 'no_of_reviews', 'drug_link',
       'medical_condition_url'],
      dtype='object')

In [6]:
df.shape


(2966, 17)

In [7]:
#check missing values
df.isnull().sum()

drug_name                           1
medical_condition                   0
side_effects                      124
generic_name                       44
drug_classes                       83
brand_names                      1214
activity                            1
rx_otc                              2
pregnancy_category                230
csa                                 1
alcohol                          1554
related_drugs                    1470
medical_condition_description       1
rating                           1361
no_of_reviews                    1361
drug_link                          35
medical_condition_url              35
dtype: int64

In [9]:
df['drug_name'].nunique()
df['medical_condition'].nunique()
df['rating'].describe()


count     1605
unique      88
top         10
freq       153
Name: rating, dtype: object

In [10]:
#create case_id
df['case_id'] = range(1, len(df) + 1)

In [11]:
#Rename PV-relevant columns
df = df.rename(columns={
    'drug_name': 'suspect_drug',
    'side_effects': 'adverse_event',
    'medical_condition': 'indication'
})

In [12]:
#Clean rating column
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

In [13]:
#Create seriousness flag
serious_terms = [
    'death', 'fatal', 'hospital', 'seizure',
    'life-threatening', 'vision loss',
    'heart attack', 'stroke'
]

df['serious_ae'] = df['adverse_event'].str.contains(
    '|'.join(serious_terms),
    case=False,
    na=False
)

In [15]:
import pandas as pd

# Load raw CSV
df = pd.read_csv("drugs_side_effects_drugs_com.csv", encoding="utf-8", engine="python")

# Select only needed columns
cols = [
    "drug_name",
    "medical_condition",
    "side_effects",
    "generic_name",
    "drug_classes",
    "rx_otc",
    "pregnancy_category",
    "rating"
]

df_clean = df[cols]

# Save clean CSV with safe quoting
df_clean.to_csv(
    "drug_safety_clean.csv",
    index=False,
    quoting=1  # csv.QUOTE_ALL
)

print("Clean CSV generated successfully")

Clean CSV generated successfully


In [6]:
# Inspect rating column values
df['rating'].unique()[:10]

array(['6.8', '7.2', '5.7', '7.9', '7.4', '7.6', '7.7', '8', '8.5', '7.8'],
      dtype=object)

## Data Readiness Summary

- Dataset successfully loaded and validated
- Missing values identified for safety-relevant fields
- Rating column observed to contain mixed data types
- Cleaned and transformed values were handled at the SQL layer
- Final analysis and aggregation performed using PostgreSQL

## AI-Assisted Observations

Using AI assistance, the following high-level observations were documented
based on SQL and dashboard outputs:

- Certain drugs show a higher frequency of reported adverse events
- Serious adverse events are concentrated in a limited subset of drugs
- Prescription-only (Rx) drugs exhibit a higher safety reporting volume than OTC
- Lower user ratings tend to align with increased reporting of serious adverse events

⚠️ AI was used only to assist in summarizing analytical findings.
Final interpretation remained analyst-driven.

## Next Steps

- Interactive dashboards and forecasting implemented in Power BI
- SQL queries used for signal detection logic
- This notebook serves as the data preparation and validation layer

## Scope Note

This notebook intentionally focuses on data validation and preparation.
Statistical analysis, trend detection, and forecasting are implemented
using SQL and Power BI to reflect real-world analytics workflows.